In [17]:
# ------------------------------------------------- #
# 1- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 2- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 3- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 4- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 14 - Hibrido cosseno
# 15 - Hibrido pearson
# ------------------------------------------------- #

# 5- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 6- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 7- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 8- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 16 - Hibrido cosseno
# 17 - Hibrido pearson
# ------------------------------------------------- #

# 9- Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
# *
# 10- Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
# *
# 11 - Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *
# 12 - Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 18 - Hibrido cosseno
# 19 - Hibrido pearson
# ------------------------------------------------- #
# 13 - multiclasse
# ------------------------------------------------- #

In [18]:
# -----------------------------
# bibliotecas -> TODO: Remover as que não uso
# -----------------------------
import random
import numpy as np
import json
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn import svm
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import os
import time
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, cohen_kappa_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# -----------------------------
# GRUPOS DE MÉTODOS
# -----------------------------
ITEM_PROFILE_METHODS = {1,2,5,6, 20, 21}
USER_PROFILE_METHODS = {3,4,7,8, 22, 23}
STRUCTURE_ONLY_METHODS = {9,10}
STRUCTURE_PERF_METHODS = {11,12,13}
HYBRID_PREFORM_METHODS = {14,15}
HYBRID_LEVEL_METHODS = {16,17}
HYBRID_STRUCTURE_METHODS = {18,19}


# -----------------------------
# PREPARANDO O DATASET
# -----------------------------

def initialize_data_frame():
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 2000)
    sns.set(rc={'figure.figsize':(19.20, 10.80)})


def get_dungeon_data_frame():
    with open('backup.json') as data_file:
        json_data = json.load(data_file)
    return pd.DataFrame(json_data['__collections__']['dungeons']).T


def get_user_data_frame():
    with open('backup.json') as data_file:
        json_data = json.load(data_file)
    return pd.DataFrame(json_data['__collections__']['users']).T


def drop_invalid_rows(_data_frame):
    _data_frame.drop(_data_frame[_data_frame['UniqueRoomsEntered'] == 0].index, inplace=True)
    return _data_frame


def get_percentages(_data_frame):
    _data_frame['LockUsage'] = _data_frame['LocksOpened'].div(_data_frame['TotalLocks'].values)*100
    _data_frame['KeyCollection'] = _data_frame['KeysCollected'].div(_data_frame['TotalKeys'].values)*100
    _data_frame['MapCompletion'] = _data_frame['UniqueRoomsEntered'].div(_data_frame['TotalRooms'].values)*100
    _data_frame['RoomRevisitRate'] = _data_frame['RoomsEntered'].div(_data_frame['TotalRooms'].values)*100
    _data_frame['EnemyKillRate'] = _data_frame['EnemiesKilled'].div(_data_frame['TotalEnemies'].values)*100
    _data_frame['TotalAttempts'] = _data_frame['TotalWins'] + _data_frame['TotalDeaths'].values
    _data_frame['TotalAttempts'] = _data_frame['TotalAttempts'].astype(float)
    _data_frame['TotalWins'] = _data_frame['TotalWins'].astype(float)
    _data_frame['Success Rate'] = _data_frame['TotalAttempts'].div(_data_frame['TotalWins'].values)*100
    _data_frame.replace([np.inf, -np.inf], 0, inplace=True)
    return _data_frame


def split_preform_answers(_data_frame):
    _data_frame[['A', 'C', 'I', 'M']] = _data_frame['PreFormAnswers'].str.split(',', expand=True)
    _data_frame['A'] = _data_frame['A'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['C'] = _data_frame['C'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['I'] = _data_frame['I'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['M'] = _data_frame['M'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    return _data_frame

# -----------------------------
# DATA AUGMENTATION
# -----------------------------

def augment_dataframe_with_scaling(df, y, n, scale_factor):
    """
    Augments the DataFrame by adding `n` new rows, scaling numeric columns by a given factor.
    The new rows inherit the same 'y' values as their corresponding sampled rows.

    Parameters:
        df (pd.DataFrame): Original DataFrame with numeric columns and a 'y' column.
        n (int): Number of new rows to generate.
        scale_factor (float): The factor by which numeric columns will be scaled.

    Returns:
        pd.DataFrame: DataFrame with the original and new rows.
    """
    # Identify numeric columns, excluding 'y'
    numeric_columns = df.select_dtypes(include=['number']).columns.difference(y)
    # Randomly sample `n` rows from the original DataFrame
    sampled_rows = df.sample(n=n, replace=True).reset_index(drop=True)

    # Scale numeric columns
    for col in numeric_columns:
        sampled_rows[col] *= scale_factor

    # Append the scaled rows to the original DataFrame
    augmented_df = pd.concat([df, sampled_rows], ignore_index=True)

    return augmented_df


def augment_data_random(_df, y, n):
    """
    Augments the DataFrame by adding `n` new rows.
    - Adds noise to numerical columns using a normal distribution.
    - Retains the same 'y' values for the newly added rows as their corresponding sampled rows.

    Parameters:
        _df (pd.DataFrame): Original DataFrame with numeric columns and a 'y' column.
        y (list): the column(s) that will have their value retained
        n (int): Number of new rows to generate.

    Returns:
        pd.DataFrame: DataFrame with the original and new rows.
    """
    # Identify float and integer columns, excluding 'y'
    float_columns = _df.select_dtypes(include=['float']).columns.difference(y)
    int_columns = _df.select_dtypes(include=['integer']).columns.difference(y)

    # Randomly sample `n` rows from the original DataFrame
    sampled_rows = _df.sample(n=n, replace=True).reset_index(drop=True)

    # Add noise to float columns
    for col in float_columns:
        if random.randint(1, 100) > 50:
            std = _df[col].std()
            noise = np.random.normal(loc=0, scale=std, size=n)
            sampled_rows[col] += noise

    for col in int_columns:
        if random.randint(1, 100) > 50:
            std = _df[col].std()
            noise = np.random.normal(loc=0, scale=std, size=n)
            sampled_rows[col] += noise

    # Ensure integer columns remain integers
    sampled_rows[int_columns] = sampled_rows[int_columns].round().astype(int)

    # Append the sampled (augmented) rows to the original DataFrame
    augmented_df = pd.concat([_df, sampled_rows], ignore_index=True)
    return augmented_df

def leave_one_out_split(df, user_col='PlayerId'):
    train_list, test_list = [], []
    for u, grp in df.groupby(user_col):
        if len(grp)<2:
            train_list.append(grp); continue
        test_list.append(grp.sample(1, random_state=42))
        train_list.append(grp.drop(test_list[-1].index))
    return pd.concat(train_list), pd.concat(test_list)

In [19]:
# -----------------------------
# ESTRUTURA
# -----------------------------

def build_collab_df(_df, method, item_col_pass='PostFormAnswers'):
    # ----------------------------
    # DEFINIÇÃO DE COLUNAS E ITEM
    # ----------------------------
    if method in ITEM_PROFILE_METHODS:
      item_col = 'PreFormAnswers' if method in {1, 2} else 'LevelName'
      cols = ['PlayerId', item_col, 'PostFormAnswers']
      rating_type = "postform"

    elif method in USER_PROFILE_METHODS:
        item_col = 'PreFormAnswers' if method in {3, 4} else 'LevelName'
        cols = ['PlayerId', item_col,
                'Success Rate', 'TotalLostHealth',
                'EnemyKillRate', 'MapCompletion']
        rating_type = "performance"

    elif method in STRUCTURE_ONLY_METHODS.union(STRUCTURE_PERF_METHODS):
        cols = ['PlayerId', 'TotalRooms', 'TotalKeys']
        rating_type = "structure_only" if method in STRUCTURE_ONLY_METHODS else "structure_perf"

        if rating_type == "structure_only":
            cols.append('PostFormAnswers')
        else:
            cols += ['Success Rate', 'TotalLostHealth',
                     'EnemyKillRate', 'MapCompletion']

    elif method in HYBRID_PREFORM_METHODS.union(HYBRID_LEVEL_METHODS):
        # Define qual coluna será o Item
        item_col = 'PreFormAnswers' if method in HYBRID_PREFORM_METHODS else 'LevelName'

        cols = ['PlayerId', item_col, 'PostFormAnswers',
                'Success Rate', 'TotalLostHealth',
                'EnemyKillRate', 'MapCompletion']

        rating_type = "hybrid"

    elif method in HYBRID_STRUCTURE_METHODS:
        cols = ['PlayerId', 'TotalRooms', 'TotalKeys',
                'RoomsEntered', 'UniqueRoomsEntered',
                'PostFormAnswers', 'Success Rate',
                'TotalLostHealth', 'EnemyKillRate',
                'MapCompletion']
        rating_type = "hybrid_structure"

    else:
        raise ValueError(f"Método {method} inválido.")

    df = _df[cols].copy()
    df = df.rename(columns={'PlayerId': 'User'})

    if rating_type in ["structure_only", "structure_perf", "hybrid_structure"]:
        df['Item'] = build_item_from_structure(df)

    elif rating_type == "hybrid":
        df = df.rename(columns={item_col: 'Item'})

    else:
        df = df.rename(columns={item_col: 'Item'})

    RATING_STRATEGIES = {
        "postform": lambda df: df['PostFormAnswers'].apply(mean_post_forms),
        "structure_only": lambda df: df['PostFormAnswers'].apply(mean_post_forms),
        "performance": compute_performance_rating,
        "structure_perf": compute_performance_rating,
        "hybrid": compute_hybrid_rating,
        "hybrid_structure": compute_hybrid_rating
    }

    df['Rating'] = minmax(
        RATING_STRATEGIES[rating_type](df)
    )

    return df[['User', 'Item', 'Rating']]

In [20]:
# -----------------------------
# PREDICT
# -----------------------------
def weighted_mean(similarities, ratings):
    numerator = (similarities * ratings).sum()
    denominator = similarities.abs().sum()
    return numerator / denominator if denominator != 0 else 0

def predict_cf(user, item, user_item_matrix, similarity_df, k=20, mode="user"):
          if mode == "user":
            if user not in similarity_df.index or item not in user_item_matrix.columns:
                return 0

            similar_entities = similarity_df.loc[user].drop(user, errors="ignore")
            ratings = user_item_matrix[item]

          elif mode == "item":
            if item not in similarity_df.columns or user not in user_item_matrix.index:
                return 0

            similar_entities = similarity_df[item]
            ratings = user_item_matrix.loc[user]

          else:
            raise ValueError("mode deve ser 'user' ou 'item'")
          # 👉 Filtrar itens não avaliados pelo usuário
          mask = ratings  != 0
          similar_entities = similar_entities[mask];
          ratings = ratings[mask]
          if similar_entities.empty:
              return 0
          # (Opcional) usar apenas os top-K vizinhos
          similar_entities  = similar_entities .sort_values(ascending=False).head(k)
          ratings = ratings.loc[similar_entities .index]

          return weighted_mean(similar_entities, ratings)

def predict_hybrid(user, item,
                   user_item_matrix,
                   item_similarity_df,
                   user_similarity_df,
                   alpha=0.5,k=20):

    score_item = predict_cf(
        user, item,
        user_item_matrix,
        item_similarity_df,
        mode="item",
        k=k
    )

    score_user = predict_cf(
        user, item,
        user_item_matrix,
        user_similarity_df,
        mode="user",
        k=k
    )

    return alpha * score_item + (1 - alpha) * score_user

In [21]:
# -----------------------------
# RECOMENDAÇÃO
# -----------------------------
def recommend_from_other_user(target_user, other_user, user_item_matrix):

    print(f"\nRecomendando para usuário {target_user} com base no usuário {other_user}")

    # Itens que target_user já jogou
    target_items = user_item_matrix.loc[target_user]

    # Itens que other_user já jogou
    other_items = user_item_matrix.loc[other_user]

    # Pegar apenas itens que:
    # - other_user jogou (rating > 0)
    # - target_user NÃO jogou (rating == 0)
    candidate_items = other_items[
        (other_items > 0) & (target_items == 0)
    ]

    if len(candidate_items) == 0:
        print("Nenhum item novo para recomendar.")
        return

    # Ordenar pelos melhores ratings do other_user
    recommendations = candidate_items.sort_values(ascending=False)

    print("\nItens recomendados:")
    print(recommendations.head(5))

# ---------------- RECOMENDAÇÃO USER-BASED EXPLICÁVEL ---------------- #

def recommend_user_based_explain(target_user, user_item_matrix, user_similarity_df, top_n=5):

    print(f"\nRecomendações para usuário {target_user}")

    # Similaridade do usuário alvo com todos os outros
    similarities = user_similarity_df.loc[target_user].drop(target_user)
    # Pegar itens que o usuário ainda NÃO jogou
    user_ratings = user_item_matrix.loc[target_user]
    items_not_played = user_ratings[user_ratings == 0].index

    predictions = {}
    explanations = {}

    for item in items_not_played:
        # Usuários que avaliaram esse item
        users_who_rated = user_item_matrix[user_item_matrix[item] > 0].index

        numerator = 0
        denominator = 0
        contributors = []

        for other_user in users_who_rated:

            sim = similarities.get(other_user, 0)

            if sim > 0:# só similaridade positiva
                rating = user_item_matrix.loc[other_user, item]
                contribution = sim * rating

                numerator += contribution
                denominator += abs(sim)

                contributors.append({
                    "user": other_user,
                    "similarity": sim,
                    "rating": rating,
                    "contribution": contribution
                })

        if denominator > 0:
            predicted_rating = numerator / denominator
            predictions[item] = predicted_rating
            explanations[item] = contributors

    if len(predictions) == 0:
        print("Nenhum item possível para recomendar.")
        return {}

    recommendations = (
        pd.Series(predictions)
        .sort_values(ascending=False)
        .head(top_n)
    )

    # Mostrar explicação detalhada
    for item in recommendations.index:
        print("\n---------------------------------------")
        print(f"Explicação para {item}")
        print("---------------------------------------")

        contribs = sorted(
            explanations[item],
            key=lambda x: x["contribution"],
            reverse=True
        )

        for c in contribs[:5]:  # top 5 contribuidores
            print(
                f"Usuário {c['user']} | "
                f"similaridade={round(c['similarity'],3)} | "
                f"nota={round(c['rating'],2)} | "
                f"contribuição={round(c['contribution'],3)}"
            )
    return {item: explanations[item] for item in recommendations.index}
# Recommend items similar to 'Item1'
def recommend_similar_items(item, similarity_df, top_n=3):
  return similarity_df[item].sort_values(ascending=False)[1:top_n+1]

def print_user_ratings(user_item_matrix, user_similarity_df, user1_idx=0, user2_idx=167):
    # seleciona usuários
    u1 = user_similarity_df.index[user1_idx]
    u2 = user_similarity_df.index[user2_idx]

    print("Notas do usuário 1:")
    print(user_item_matrix.loc[u1].dropna())

    print("\nNotas do usuário 2:")
    print(user_item_matrix.loc[u2].dropna())

def recommend_hybrid(user,
                     user_item_matrix,
                     item_similarity_df,
                     user_similarity_df,
                     top_n=5,
                     alpha=0.5):

    items = user_item_matrix.columns
    predictions = {}

    for item in items:
        if user_item_matrix.loc[user, item] == 0:  # não avaliou
            predictions[item] = predict_hybrid(
                user,
                item,
                user_item_matrix,
                item_similarity_df,
                user_similarity_df,
                alpha
            )

    ranked = sorted(predictions.items(),
                    key=lambda x: x[1],
                    reverse=True)

    return ranked[:top_n]

In [22]:
# -----------------------------
# ANÁLISES
# -----------------------------
def inspect_dataset(user_frame, dungeon_frame):
    profiles = user_frame['PreFormAnswers'].apply(tuple)

    print("Combinações únicas:")
    print(sorted(profiles.unique()))

    print("\nQuantidade de combinações únicas:")
    print(profiles.nunique())

    print("\nFrequência de cada combinação:")
    print(profiles.value_counts())

    dungeon_frame = pd.merge(
        dungeon_frame,
        user_frame[['PlayerId', 'PreFormAnswers']],
        on=['PlayerId'],
        how='left'
    )

    print("\nFrequência no dungeon_frame final:")
    print(dungeon_frame['PreFormAnswers'].value_counts())

    # Listar todas as colunas
    colunas = dungeon_frame.columns.tolist()

    print(f"\nTotal de colunas: {len(colunas)}\n")

    # Exibir em ordem alfabética numerada
    for i, col in enumerate(sorted(colunas), 1):
        print(f"{i:3d}. {col}")

def analyze_user_item_matrix(user_item_matrix):

    print("------------- VERIFICAÇÃO DOS DADOS -------------")
    print("Número de usuários:", user_item_matrix.shape[0])
    print("Número de itens:", user_item_matrix.shape[1])

    print("\nQuantidade média de avaliações por usuário:")
    print(user_item_matrix.count(axis=1).mean())

    print("\nQuantidade média de usuários por item:")
    print(user_item_matrix.count(axis=0).mean())

    print("\n------------- VARIÂNCIA ITEM POR ITEM -------------")
    print("Variância de cada item:")
    print(user_item_matrix.var())

In [23]:
# -----------------------------
# CÁLCULOS
# -----------------------------
def evaluate_rating_metrics(predictions_df, test_matrix):

    y_true = []
    y_pred = []

    for _, row in predictions_df.iterrows():
        user = row["User"]
        item = row["Item"]

        if user in test_matrix.index and item in test_matrix.columns:
            true_rating = test_matrix.loc[user, item]
            if true_rating != 0:
                y_true.append(true_rating)
                y_pred.append(row["Prediction"])

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    return rmse, mae

def compute_rmse(test_df, prediction_col, rating_col):
    """
    Calcula RMSE ignorando:
        - valores NaN
        - casos sem predição
        - divisão por zero
    """

    # Manter apenas linhas com valor real e predito
    mask = (
        test_df[rating_col].notna() &
        test_df[prediction_col].notna()
    )

    y_true = test_df.loc[mask, rating_col].values
    y_pred = test_df.loc[mask, prediction_col].values

    if len(y_true) == 0:
        raise ValueError("Nenhuma predição válida para calcular RMSE.")

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    return rmse

def precision_at_k(rec_df, test_df, k=5, threshold=1.0):
    precision_list = []

    for user in test_df["User"].unique():

        top_k = rec_df[rec_df["User"] == user].head(k)["Item"].values
        true_items = test_df[(test_df["User"] == user) & (test_df["Rating"] >= threshold)]["Item"].values

        if len(true_items) == 0:
            continue

        hits = len(set(top_k) & set(true_items))
        precision_list.append(hits / k)

    return np.mean(precision_list)
def recall_at_k(rec_df, test_df, k=5, threshold=1.0):
    recall_list = []

    for user in test_df["User"].unique():

        top_k = rec_df[rec_df["User"] == user].head(k)["Item"].values
        true_items = test_df[(test_df["User"] == user) & (test_df["Rating"] >= threshold)]["Item"].values

        if len(true_items) == 0:
            continue

        hits = len(set(top_k) & set(true_items))
        recall_list.append(hits / len(true_items))

    return np.mean(recall_list)
def hit_rate_at_k(rec_df, test_df, k=5):
    hits = 0
    users = test_df["User"].unique()

    for user in users:
        top_k = rec_df[rec_df["User"] == user].head(k)["Item"].values
        true_items = test_df[test_df["User"] == user]["Item"].values

        if len(set(top_k) & set(true_items)) > 0:
            hits += 1

    return hits / len(users)
def mrr(rec_df, test_df):
    rr_total = []

    for user in test_df["User"].unique():

        top_k = rec_df[rec_df["User"] == user].reset_index()

        true_items = set(test_df[test_df["User"] == user]["Item"])

        rank = 0

        for i, row in top_k.iterrows():
            if row["Item"] in true_items:
                rank = i + 1
                break

        if rank > 0:
            rr_total.append(1 / rank)

    return np.mean(rr_total)

def similiarity_pearson(user_item_matrix):
    # Pearson item-item
    item_similarity_pearson = user_item_matrix.corr(method='pearson')

    print("\nITEM POR ITEM (PEARSON):\n")
    print(item_similarity_pearson)
    item_similarity_df = item_similarity_pearson

    user_similarity_pearson = user_item_matrix.T.corr(method='pearson')
    user_similarity_pearson = user_similarity_pearson.fillna(0)

    print("\nUSUARIO POR USUARIO (PEARSON):\n")
    print(user_similarity_pearson)

    user_similarity_df  = user_similarity_pearson

    return item_similarity_df, user_similarity_df

def similiarity_cosine(user_item_matrix):
    # Compute item similarity using cosine similarity
    item_similarity = cosine_similarity(user_item_matrix.T)
    item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)

    print("\nITEM POR ITEM (COSSENO)):\n")
    print(item_similarity_df)

    #centralizando
    user_item_centered = user_item_matrix.sub(user_item_matrix.mean(axis=1), axis=0)
    #print("\nconsseno cetnralizado:\n")
    #print(user_item_centered)

    # Similaridade entre itens
    # item por item
    item_similarity_centered = cosine_similarity(user_item_centered.T)
    item_similarity_centered_df = pd.DataFrame(item_similarity_centered, index=user_item_centered.columns, columns=user_item_centered.columns)

    #print("\nconsseno:\n")
    #print(item_similarity_df)

    #print("\nconsseno cetnralizado:\n")
    #print(item_similarity_centered)

    # usuario por usuário
    user_similarity = cosine_similarity(user_item_centered)
    user_similarity_df = pd.DataFrame( user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index )
    #print("\nUSUARIO POR USUARIO:\n")
    #print(user_similarity_df.iloc[:50, :50])

    return item_similarity_df, user_similarity_df

def compute_performance_rating(df):
    df['TotalLostHealth_norm'] = minmax(df['TotalLostHealth'])

    return (
        df['Success Rate']
        - df['TotalLostHealth_norm']
        + df['EnemyKillRate']
        + df['MapCompletion']
    )

def build_item_from_structure(df):
    df['TotalRooms'] = df['TotalRooms'].replace(0, 1)

    return (
        "R" + df['TotalRooms'].astype(str) +
        "_K" + df['TotalKeys'].astype(str))# +        "_RR" + (df['RoomsEntered'] - df['UniqueRoomsEntered']).astype(str)    )

def compute_hybrid_rating(df, alpha=0.5):
    df_valid = df['PostFormAnswers'].apply(mean_post_forms)
    df_comp = compute_performance_rating(df)

    return alpha * df_comp + (1 - alpha) * df_valid

def mean_post_forms(x):
    return sum(x)/len(x) if isinstance(x, list) else x

def minmax(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

In [24]:
def recomendando(method, user_item_matrix, user_similarity_df, item_similarity_df,_df,df_original):
  # ------------- RECOMENDAR ITEM POR ITEM ------------#
      # esta comentado, pq nao fareirpos oturos casos
      if(method >=1 and method <=4):
        # Example recommendation
        similar_items = recommend_similar_items('A=4, C=3, I=2, M=1', item_similarity_df)
        print("\nItems similar to A=4, C=3, I=2, M=1:", similar_items)

      # Recomendar todos os itens por similarida itens
      print("--------------------6----------------------------")
      all_recommendations = {}

      for item in user_item_matrix.columns:
          all_recommendations[item] = recommend_similar_items(item, item_similarity_df)

      print(all_recommendations)

      # ------------- SIMILIARIDA USUARIO POR USUARIO ------------#
      print("--------------------7----------------------------")

      # print abaixo é pra ENTENDNEO O RESULTADO ACIMA
      print_user_ratings(user_item_matrix, user_similarity_df)

      # ------------- testando um usuario com mais de um nivel ------------#
      print("--------------------8----------------------------")
      #analyze_user_activity(user_item_matrix) # -> TODO; Descobri onde que tivrei e voltar

      # Recomendar para u1 baseado em u2
      print("--------------------9----------------------------")
      recommend_from_other_user(user_item_matrix.index[0], user_item_matrix.index[167], user_item_matrix)

      # Recomendar para u2 baseado em u1
      print("--------------------10----------------------------")
      recommend_from_other_user(user_item_matrix.index[167], user_item_matrix.index[0], user_item_matrix)

      # Escolher usuário
      print("--------------------13----------------------------")
      recs = recommend_user_based_explain(user_item_matrix.index[0], user_item_matrix, user_similarity_df, top_n=5)

      # ---- NOVO: TABELA DE INIMIGOS DOS NÍVEIS RECOMENDADOS ----
            # ---- TABELA COM MÉDIAS DOS NÍVEIS RECOMENDADOS ----
                # -------- TABELA DE NPCS --------
                # ========== TABELA DE PROGRESSO PARA NÍVEIS RECOMENDADOS ==========
      if not recs:
        print("Nenhuma recomendação disponível.")
        return None

      # Executar todas as análises e capturar os valores retornados
      rooms_medio, keys_medio, locks_medio = display_structure_analysis(df_original, recs)
      quest_A, quest_C, quest_I, quest_M = display_quests_analysis(df_original, recs)
      npcs_medio, interagidos_medio = display_npcs_analysis(df_original, recs)
      enemies_medio, health_medio, deaths_medio = display_difficulty_analysis(df_original, recs)

            # ========== SUGESTÃO DE NOVA DUNGEON BASEADA NOS RANGES ==========
      print("\n" + "="*80)
      print("🏰 SUGESTÃO DE NOVA DUNGEON (BASEADA NOS RANGES DOS RECOMENDADOS)")
      print("="*80)

      # Montar sugestão
      print("\n📋 Configuração sugerida para nova dungeon:")
      print(f"  - Salas (TotalRooms): {int(round(rooms_medio))}")
      print(f"  - Chaves (TotalKeys): {int(round(keys_medio))}")
      print(f"  - Fechaduras (TotalLocks): {int(round(locks_medio))}")
      print(f"  - Inimigos (TotalEnemies): {int(round(enemies_medio))}")
      print(f"  - NPCs (TotalNpcs): {int(round(npcs_medio))}")
      print(f"  - Perfil de Quests (A% / C% / I% / M%): {quest_A:.1f}% / {quest_C:.1f}% / {quest_I:.1f}% / {quest_M:.1f}%")

      # 🔥 MULTI-VIEW RECOMMENDATION (NOVO)
      print("--------------------14----------------------------")
      if method == 13:
          print("\n\n=========== MULTI VIEW ===========")
      elif (method >= 14):
          print("\n=========== HYBRID RECOMMENDATION ===========")
          recommend_hybrid(user_item_matrix.index[0], user_item_matrix, item_similarity_df, user_similarity_df)

In [25]:
def display_structure_analysis(df_original, recs):
    """Análise de estrutura (salas, chaves, fechaduras) para níveis recomendados e similares"""
    if not recs:
        print("Nenhuma recomendação disponível.")
        return None, None, None

    # ========== TABELA DE ESTRUTURA PARA NÍVEIS RECOMENDADOS ==========
    print("\n" + "="*80)
    print("📊 ESTRUTURA - MÉDIAS PARA NÍVEIS RECOMENDADOS")
    print("="*80)
    print(f"{'Nível':<30} {'Salas (TotalRooms)':<20} {'Chaves (TotalKeys)':<20} {'Fechaduras (TotalLocks)':<20}")
    print("-"*80)

    for nivel in recs.keys():
        nivel_data = df_original[df_original['LevelName'] == nivel]
        if len(nivel_data) > 0:
            rooms = nivel_data['TotalRooms'].mean()
            keys = nivel_data['TotalKeys'].mean()
            locks = nivel_data['TotalLocks'].mean()
            print(f"{nivel:<30} {rooms:<20.1f} {keys:<20.1f} {locks:<20.1f}")
        else:
            print(f"{nivel:<30} {'N/A':<20} {'N/A':<20} {'N/A':<20}")

    # ========== TABELA DE ESTRUTURA PARA TODOS OS NÍVEIS ==========
    df_todos_estrutura = df_original.groupby('LevelName').agg({
        'TotalRooms': 'mean',
        'TotalKeys': 'mean',
        'TotalLocks': 'mean'
    }).reset_index()

    # Classificação baseada em TotalRooms
    def classificar_estrutura(row):
        if row['TotalRooms'] <= 10:
            return "Pequena"
        elif row['TotalRooms'] <= 20:
            return "Média"
        else:
            return "Grande"

    df_todos_estrutura['ClasseEstrutura'] = df_todos_estrutura.apply(classificar_estrutura, axis=1)

    # ========== OPÇÃO A: MESMA CLASSE DE ESTRUTURA ==========
    find_same_structure_class(df_todos_estrutura, recs)

    # ========== OPÇÃO B: RANGE BASEADO NOS RECOMENDADOS ==========
    rooms_medio, keys_medio, locks_medio = find_similar_structure_range(df_todos_estrutura, recs)

    return rooms_medio, keys_medio, locks_medio


def find_same_structure_class(df_todos_estrutura, recs):
    """Encontra níveis com a mesma classe de estrutura dos recomendados"""
    classe_estrutura_alvo = df_todos_estrutura[df_todos_estrutura['LevelName'].isin(recs.keys())].iloc[0]['ClasseEstrutura']
    niveis_mesma_classe_estrutura = df_todos_estrutura[
        (df_todos_estrutura['ClasseEstrutura'] == classe_estrutura_alvo) &
        (~df_todos_estrutura['LevelName'].isin(recs.keys()))
    ]

    print(f"\n🔍 Outros níveis com classe de estrutura '{classe_estrutura_alvo}':")
    if len(niveis_mesma_classe_estrutura) > 0:
        print(niveis_mesma_classe_estrutura[['LevelName', 'TotalRooms', 'TotalKeys', 'TotalLocks']].head(10).to_string(index=False))
    else:
        print("Nenhum nível encontrado.")


def find_similar_structure_range(df_todos_estrutura, recs):
    """Encontra níveis dentro do range de estrutura dos recomendados e retorna valores médios"""
    rec_stats_estrutura = df_todos_estrutura[df_todos_estrutura['LevelName'].isin(recs.keys())]
    min_rooms = rec_stats_estrutura['TotalRooms'].min()
    max_rooms = rec_stats_estrutura['TotalRooms'].max()
    min_keys = rec_stats_estrutura['TotalKeys'].min()
    max_keys = rec_stats_estrutura['TotalKeys'].max()
    min_locks = rec_stats_estrutura['TotalLocks'].min()
    max_locks = rec_stats_estrutura['TotalLocks'].max()

    range_rooms = (min_rooms, max_rooms)
    range_keys = (min_keys, max_keys)
    range_locks = (min_locks, max_locks)

    niveis_no_range_estrutura = df_todos_estrutura[
        (df_todos_estrutura['TotalRooms'] >= range_rooms[0]) &
        (df_todos_estrutura['TotalRooms'] <= range_rooms[1]) &
        (df_todos_estrutura['TotalKeys'] >= range_keys[0]) &
        (df_todos_estrutura['TotalKeys'] <= range_keys[1]) &
        (df_todos_estrutura['TotalLocks'] >= range_locks[0]) &
        (df_todos_estrutura['TotalLocks'] <= range_locks[1]) &
        (~df_todos_estrutura['LevelName'].isin(recs.keys()))
    ]

    print(f"\n🔍 Outros níveis dentro do range de Estrutura dos recomendados:")
    print(f"   Salas: {range_rooms[0]:.1f} a {range_rooms[1]:.1f}")
    print(f"   Chaves: {range_keys[0]:.1f} a {range_keys[1]:.1f}")
    print(f"   Fechaduras: {range_locks[0]:.1f} a {range_locks[1]:.1f}")

    if len(niveis_no_range_estrutura) > 0:
        print(niveis_no_range_estrutura[['LevelName', 'TotalRooms', 'TotalKeys', 'TotalLocks']].head(10).to_string(index=False))
    else:
        print("Nenhum nível encontrado.")

    # Calcular valores médios do range
    rooms_medio = (range_rooms[0] + range_rooms[1]) / 2
    keys_medio = (range_keys[0] + range_keys[1]) / 2
    locks_medio = (range_locks[0] + range_locks[1]) / 2

    return rooms_medio, keys_medio, locks_medio


def display_quests_analysis(df_original, recs):
    """Análise de quests (perfis A, C, I, M) para níveis recomendados e similares"""
    if not recs:
        print("Nenhuma recomendação disponível.")
        return None, None, None, None

    # ========== QUESTS - USANDO COLUNAS "COMPLETED" ==========
    quest_cols = [
        'CompletedAchievementQuests',
        'CompletedCreativityQuests',
        'CompletedImmersionQuests',
        'CompletedMasteryQuests'
    ]
    quest_cols = [c for c in quest_cols if c in df_original.columns]

    # Função para calcular o perfil percentual (A, C, I, M)
    def calc_profile(row):
        total = sum(row[col] for col in quest_cols)
        if total == 0:
            return pd.Series([0, 0, 0, 0])
        a = row['CompletedAchievementQuests'] / total
        c = row['CompletedCreativityQuests'] / total
        i = row['CompletedImmersionQuests'] / total
        m = row['CompletedMasteryQuests'] / total
        return pd.Series([a, c, i, m])

    # ========== TABELA DE QUESTS PARA NÍVEIS RECOMENDADOS ==========
    print("\n" + "="*80)
    print("📊 QUESTS - PERFIS PERCENTUAIS (A, C, I, M) PARA NÍVEIS RECOMENDADOS")
    print("="*80)
    print(f"{'Nível':<30} {'A% (Achievement)':<18} {'C% (Creativity)':<18} {'I% (Immersion)':<18} {'M% (Mastery)':<18}")
    print("-"*80)

    for nivel in recs.keys():
        nivel_data = df_original[df_original['LevelName'] == nivel]
        if len(nivel_data) > 0:
            means = nivel_data[quest_cols].mean()
            total = means.sum()
            if total == 0:
                print(f"{nivel:<30} {'N/A':<18} {'N/A':<18} {'N/A':<18} {'N/A':<18}")
                continue
            a = (means['CompletedAchievementQuests'] / total) * 100
            c = (means['CompletedCreativityQuests'] / total) * 100
            i = (means['CompletedImmersionQuests'] / total) * 100
            m = (means['CompletedMasteryQuests'] / total) * 100
            print(f"{nivel:<30} {a:<17.1f}% {c:<17.1f}% {i:<17.1f}% {m:<17.1f}%")
        else:
            print(f"{nivel:<30} {'N/A':<18} {'N/A':<18} {'N/A':<18} {'N/A':<18}")

    # ========== PERFIS PARA TODOS OS NÍVEIS ==========
    df_todos_quests = df_original.groupby('LevelName')[quest_cols].mean().reset_index()
    df_todos_quests[['A_pct', 'C_pct', 'I_pct', 'M_pct']] = df_todos_quests.apply(calc_profile, axis=1)

    # Calcular o perfil DOMINANTE
    def get_dominant(row):
        vals = {'A': row['A_pct'], 'C': row['C_pct'], 'I': row['I_pct'], 'M': row['M_pct']}
        return max(vals, key=vals.get) if sum(vals.values()) > 0 else 'N/A'

    df_todos_quests['PerfilDominante'] = df_todos_quests.apply(get_dominant, axis=1)

    # ========== OPÇÃO A: MESMO PERFIL DOMINANTE ==========
    find_same_quest_profile(df_todos_quests, recs)

    # ========== OPÇÃO B: PERFIS SIMILARES ==========
    quest_A, quest_C, quest_I, quest_M = find_similar_quest_profiles(df_todos_quests, recs)

    return quest_A, quest_C, quest_I, quest_M


def find_same_quest_profile(df_todos_quests, recs):
    """Encontra níveis com o mesmo perfil dominante dos recomendados"""
    perfil_alvo = df_todos_quests[df_todos_quests['LevelName'].isin(recs.keys())].iloc[0]['PerfilDominante']
    niveis_mesmo_perfil = df_todos_quests[
        (df_todos_quests['PerfilDominante'] == perfil_alvo) &
        (~df_todos_quests['LevelName'].isin(recs.keys()))
    ]

    print(f"\n🔍 Outros níveis com perfil dominante '{perfil_alvo}':")
    if len(niveis_mesmo_perfil) > 0:
        print(niveis_mesmo_perfil[['LevelName', 'A_pct', 'C_pct', 'I_pct', 'M_pct']].head(10).to_string(index=False))
    else:
        print("Nenhum nível encontrado.")


def find_similar_quest_profiles(df_todos_quests, recs):
    """Encontra níveis com perfis similares usando similaridade cosseno e retorna perfil médio"""
    rec_quests = df_todos_quests[df_todos_quests['LevelName'].isin(recs.keys())]
    target_profile = rec_quests[['A_pct', 'C_pct', 'I_pct', 'M_pct']].mean().values.reshape(1, -1)

    outros = df_todos_quests[~df_todos_quests['LevelName'].isin(recs.keys())].copy()
    if len(outros) > 0:
        profile_matrix = outros[['A_pct', 'C_pct', 'I_pct', 'M_pct']].values
        similarities = cosine_similarity(target_profile, profile_matrix)[0]
        outros['Similaridade'] = similarities

        niveis_similares = outros.sort_values('Similaridade', ascending=False).head(10)

        print(f"\n🔍 Outros níveis com perfil mais similar (por similaridade cosseno):")
        print(niveis_similares[['LevelName', 'A_pct', 'C_pct', 'I_pct', 'M_pct', 'Similaridade']].head(10).to_string(index=False))
    else:
        print("Nenhum outro nível para comparar.")

    # Calcular perfil médio em porcentagem
    quest_A = target_profile[0][0] * 100
    quest_C = target_profile[0][1] * 100
    quest_I = target_profile[0][2] * 100
    quest_M = target_profile[0][3] * 100

    return quest_A, quest_C, quest_I, quest_M


def display_npcs_analysis(df_original, recs):
    """Análise de NPCs para níveis recomendados e similares"""
    if not recs:
        print("Nenhuma recomendação disponível.")
        return None, None

    # ========== TABELA DE NPCS PARA NÍVEIS RECOMENDADOS ==========
    print("\n" + "="*80)
    print("📊 NPCS - MÉDIAS PARA NÍVEIS RECOMENDADOS")
    print("="*80)
    print(f"{'Nível':<30} {'Total NPCs':<15} {'Interagidos':<15} {'Taxa Interação':<15}")
    print("-"*80)

    for nivel in recs.keys():
        nivel_data = df_original[df_original['LevelName'] == nivel]
        if len(nivel_data) > 0:
            total_npcs = nivel_data['TotalNpcs'].mean()
            interagidos = nivel_data['NpcsInteracted'].mean()
            taxa = (interagidos / total_npcs) * 100 if total_npcs > 0 else 0
            print(f"{nivel:<30} {total_npcs:<15.1f} {interagidos:<15.1f} {taxa:<15.1f}%")
        else:
            print(f"{nivel:<30} {'N/A':<15} {'N/A':<15} {'N/A':<15}")

    # ========== TABELA DE NPCS PARA TODOS OS NÍVEIS ==========
    df_todos_npcs = df_original.groupby('LevelName').agg({
        'TotalNpcs': 'mean',
        'NpcsInteracted': 'mean'
    }).reset_index()

    df_todos_npcs['TaxaInteracao'] = (df_todos_npcs['NpcsInteracted'] / df_todos_npcs['TotalNpcs']) * 100
    df_todos_npcs['TaxaInteracao'] = df_todos_npcs['TaxaInteracao'].fillna(0)

    def classificar_npcs(row):
        if row['TotalNpcs'] <= 3:
            return "Poucos NPCs"
        elif row['TotalNpcs'] <= 8:
            return "Médios NPCs"
        else:
            return "Muitos NPCs"

    df_todos_npcs['ClasseNPC'] = df_todos_npcs.apply(classificar_npcs, axis=1)

    # ========== OPÇÃO A: MESMA CLASSE DE NPC ==========
    find_same_npc_class(df_todos_npcs, recs)

    # ========== OPÇÃO B: RANGE BASEADO NOS RECOMENDADOS ==========
    npcs_medio, interagidos_medio = find_similar_npc_range(df_todos_npcs, recs)

    return npcs_medio, interagidos_medio


def find_same_npc_class(df_todos_npcs, recs):
    """Encontra níveis com a mesma classe de NPC dos recomendados"""
    classe_npc_alvo = df_todos_npcs[df_todos_npcs['LevelName'].isin(recs.keys())].iloc[0]['ClasseNPC']
    niveis_mesma_classe_npc = df_todos_npcs[
        (df_todos_npcs['ClasseNPC'] == classe_npc_alvo) &
        (~df_todos_npcs['LevelName'].isin(recs.keys()))
    ]

    print(f"\n🔍 Outros níveis com classe NPC '{classe_npc_alvo}':")
    if len(niveis_mesma_classe_npc) > 0:
        print(niveis_mesma_classe_npc[['LevelName', 'TotalNpcs', 'NpcsInteracted', 'TaxaInteracao']].head(10).to_string(index=False))
    else:
        print("Nenhum nível encontrado.")


def find_similar_npc_range(df_todos_npcs, recs):
    """Encontra níveis dentro do range de NPCs dos recomendados e retorna valores médios"""
    rec_stats_npc = df_todos_npcs[df_todos_npcs['LevelName'].isin(recs.keys())]

    min_npcs = rec_stats_npc['TotalNpcs'].min()
    max_npcs = rec_stats_npc['TotalNpcs'].max()
    min_interagidos = rec_stats_npc['NpcsInteracted'].min()
    max_interagidos = rec_stats_npc['NpcsInteracted'].max()

    margem_npc = 0.0
    range_npcs = (min_npcs * (1 - margem_npc), max_npcs * (1 + margem_npc))
    range_interagidos = (min_interagidos * (1 - margem_npc), max_interagidos * (1 + margem_npc))

    niveis_no_range_npc = df_todos_npcs[
        (df_todos_npcs['TotalNpcs'] >= range_npcs[0]) &
        (df_todos_npcs['TotalNpcs'] <= range_npcs[1]) &
        (df_todos_npcs['NpcsInteracted'] >= range_interagidos[0]) &
        (df_todos_npcs['NpcsInteracted'] <= range_interagidos[1]) &
        (~df_todos_npcs['LevelName'].isin(recs.keys()))
    ]

    print(f"\n🔍 Outros níveis dentro do range de NPCs dos recomendados:")
    print(f"   Total NPCs: {range_npcs[0]:.1f} a {range_npcs[1]:.1f}")
    print(f"   Interagidos: {range_interagidos[0]:.1f} a {range_interagidos[1]:.1f}")

    if len(niveis_no_range_npc) > 0:
        print(niveis_no_range_npc[['LevelName', 'TotalNpcs', 'NpcsInteracted', 'TaxaInteracao']].head(10).to_string(index=False))
    else:
        print("Nenhum nível encontrado.")

    # Calcular valores médios do range
    npcs_medio = (range_npcs[0] + range_npcs[1]) / 2
    interagidos_medio = (range_interagidos[0] + range_interagidos[1]) / 2

    return npcs_medio, interagidos_medio


def display_difficulty_analysis(df_original, recs):
    """Análise de dificuldade para níveis recomendados e similares"""
    if not recs:
        print("Nenhuma recomendação disponível.")
        return None, None, None

    vida_max = df_original['TotalLostHealth'].max()
    mortes_max = df_original['TotalDeaths'].max()
    if vida_max == 0: vida_max = 1
    if mortes_max == 0: mortes_max = 1

    # ========== NÍVEIS RECOMENDADOS E ESTATÍSTICAS ==========
    print("\n" + "="*80)
    print("📊 NÍVEIS RECOMENDADOS E ESTATÍSTICAS MÉDIAS (VALORES BRUTOS)")
    print("="*80)
    print(f"{'Nível':<30} {'Inimigos':<12} {'Mortos':<12} {'Vida':<12} {'Mortes':<12} {'Dificuldade':<15} {'Classe':<15}")
    print("-"*120)

    for nivel in recs.keys():
        nivel_data = df_original[df_original['LevelName'] == nivel]
        if len(nivel_data) > 0:
            total = nivel_data['TotalEnemies'].mean()
            mortos = nivel_data['EnemiesKilled'].mean()
            vida = nivel_data['TotalLostHealth'].mean()
            mortes = nivel_data['TotalDeaths'].mean()

            kill_failure = 1 - (mortos / total) if total > 0 else 0
            vida_norm = vida / vida_max
            mortes_norm = mortes / mortes_max

            diff_score = (kill_failure * 0.2) + (vida_norm * 0.3) + (mortes_norm * 0.5)

            if diff_score < 0.3: classe = "Fácil"
            elif diff_score < 0.5: classe = "Médio"
            elif diff_score < 0.7: classe = "Difícil"
            else: classe = "Muito Difícil"

            print(f"{nivel:<30} {total:<12.1f} {mortos:<12.1f} {vida:<12.1f} {mortes:<12.2f} {diff_score:<15.2f} {classe:<15}")
        else:
            print(f"{nivel:<30} {'N/A':<20} {'N/A':<20} {'N/A':<20} {'N/A':<20}")

    # Construir as médias de TODOS os níveis do jogo
    df_todos_niveis = df_original.groupby('LevelName').agg({
        'TotalEnemies': 'mean',
        'EnemiesKilled': 'mean',
        'TotalLostHealth': 'mean',
        'TotalDeaths': 'mean'
    }).reset_index()

    def calc_dificuldade(row):
        kill_failure = 1 - (row['EnemiesKilled'] / row['TotalEnemies']) if row['TotalEnemies'] > 0 else 0
        vida_norm = row['TotalLostHealth'] / vida_max
        mortes_norm = row['TotalDeaths'] / mortes_max
        score = (kill_failure * 0.2) + (vida_norm * 0.3) + (mortes_norm * 0.5)

        if score < 0.3: classe = "Fácil"
        elif score < 0.5: classe = "Médio"
        elif score < 0.7: classe = "Difícil"
        else: classe = "Muito Difícil"
        return pd.Series([score, classe])

    df_todos_niveis[['Dificuldade', 'Classe']] = df_todos_niveis.apply(calc_dificuldade, axis=1)

    # ========== OPÇÃO A: MESMA CLASSE ==========
    find_same_difficulty_class(df_todos_niveis, recs)

    # ========== OPÇÃO B: ESTATÍSTICAS PRÓXIMAS ==========
    enemies_medio, health_medio, deaths_medio = find_similar_difficulty_stats(df_todos_niveis, recs)

    return enemies_medio, health_medio, deaths_medio


def find_same_difficulty_class(df_todos_niveis, recs):
    """Encontra níveis com a mesma classe de dificuldade dos recomendados"""
    classe_alvo = df_todos_niveis[df_todos_niveis['LevelName'].isin(recs.keys())].iloc[0]['Classe']

    niveis_similares = df_todos_niveis[
        (df_todos_niveis['Classe'] == classe_alvo) &
        (~df_todos_niveis['LevelName'].isin(recs.keys()))
    ]

    print(f"\n🔍 Outros níveis com classe '{classe_alvo}':")
    if len(niveis_similares) > 0:
        print(niveis_similares[['LevelName', 'TotalEnemies', 'TotalLostHealth', 'TotalDeaths', 'Dificuldade']].head(10).to_string(index=False))
    else:
        print("Nenhum nível encontrado.")


def find_similar_difficulty_stats(df_todos_niveis, recs):
    """Encontra níveis dentro do range estatístico dos recomendados e retorna valores médios"""
    # Pegar os dados dos níveis recomendados
    rec_stats = df_todos_niveis[df_todos_niveis['LevelName'].isin(recs.keys())]

    # Calcular min e max de cada coluna (entre os recomendados)
    min_enemies = rec_stats['TotalEnemies'].min()
    max_enemies = rec_stats['TotalEnemies'].max()
    min_health = rec_stats['TotalLostHealth'].min()
    max_health = rec_stats['TotalLostHealth'].max()
    min_deaths = rec_stats['TotalDeaths'].min()
    max_deaths = rec_stats['TotalDeaths'].max()

    # Margem (opcional)
    margem = 0.0
    range_enemies = (min_enemies * (1 - margem), max_enemies * (1 + margem))
    range_health = (min_health * (1 - margem), max_health * (1 + margem))
    range_deaths = (min_deaths * (1 - margem), max_deaths * (1 + margem))

    # Filtrar níveis que estão dentro do range (excluindo os já recomendados)
    niveis_no_range = df_todos_niveis[
        (df_todos_niveis['TotalEnemies'] >= range_enemies[0]) &
        (df_todos_niveis['TotalEnemies'] <= range_enemies[1]) &
        (df_todos_niveis['TotalLostHealth'] >= range_health[0]) &
        (df_todos_niveis['TotalLostHealth'] <= range_health[1]) &
        (df_todos_niveis['TotalDeaths'] >= range_deaths[0]) &
        (df_todos_niveis['TotalDeaths'] <= range_deaths[1]) &
        (~df_todos_niveis['LevelName'].isin(recs.keys()))
    ]

    print(f"\n🔍 Outros níveis dentro do range dos recomendados:")
    print(f"   Inimigos: {range_enemies[0]:.1f} a {range_enemies[1]:.1f}")
    print(f"   Vida:     {range_health[0]:.1f} a {range_health[1]:.1f}")
    print(f"   Mortes:   {range_deaths[0]:.2f} a {range_deaths[1]:.2f}")

    if len(niveis_no_range) > 0:
        print(niveis_no_range[['LevelName', 'TotalEnemies', 'TotalLostHealth', 'TotalDeaths', 'Dificuldade']].head(10).to_string(index=False))
    else:
        print("Nenhum nível encontrado. Tente aumentar a margem (ex: 0.2).")

    # Calcular valores médios do range
    enemies_medio = (range_enemies[0] + range_enemies[1]) / 2
    health_medio = (range_health[0] + range_health[1]) / 2
    deaths_medio = (range_deaths[0] + range_deaths[1]) / 2

    return enemies_medio, health_medio, deaths_medio

In [26]:
def run_aggregation(
    title,
    aggregation_function,
    rec_df,
    _df,
    user_profiles,
    collab_df,
    columns_to_print,
    extra_param=None,
    top_n=5
):

    if extra_param is not None:
        result = aggregation_function(
            rec_df,
            extra_param,
            user_profiles,
            collab_df,
            top_n=top_n
        )
    else:
        result = aggregation_function(
            rec_df,
            _df,
            user_profiles,
            collab_df,
            top_n=top_n
        )

    print(f"\n=== {title} ===")

    if len(result) > 0:
        print(result[columns_to_print].to_string())

    return result

def generate_recommendations(
    test_df,
    all_items,
    user_item_matrix,
    similarity_matrix,
    predict_function,
    method="user",
    alpha=0.5,
    k=20
):
    recommendations = []

    for user in test_df["User"].unique():
        for item in all_items:

            score = predict_cf(user, item, user_item_matrix, similarity_matrix,mode=method,k=k)

            recommendations.append({
                "User": user,
                "Item": item,
                "Score": score
            })

    # Converte para DataFrame
    rec_df = pd.DataFrame(recommendations)

    # Ordena
    rec_df = rec_df.sort_values(
        ["User", "Score"],
        ascending=[True, False]
    )

    return rec_df

def evaluate_error_metrics(pred_df, label=""):

    pred_df = pred_df.dropna()

    rmse = np.sqrt(mean_squared_error(
        pred_df["Rating"],
        pred_df["Prediction"]
    ))

    mae = mean_absolute_error(
        pred_df["Rating"],
        pred_df["Prediction"]
    )

    print(f"\n===== {label} ERROR METRICS =====")
    print("Predições válidas:", len(pred_df))
    print("RMSE:", rmse)
    print("MAE:", mae)

    return rmse, mae

def evaluate_ranking_metrics(rec_df, test_df, k=8, threshold=0.0, label=""):

    precision = precision_at_k(rec_df, test_df, k=k, threshold=threshold)
    recall    = recall_at_k(rec_df, test_df, k=k, threshold=threshold)

    hr  = hit_rate_at_k(rec_df, test_df, k=k)
    mrr_score = mrr(rec_df, test_df)

    print(f"\n===== {label} RANKING METRICS =====")
    print(f"Precision@{k}:", precision)
    print(f"Recall@{k}:", recall)
    print(f"HitRate@{k}:", hr)
    print("MRR:", mrr_score)

    return precision, recall, hr, mrr_score

def debug_predictions(pred_df, test_df):

    print("\n===== DEBUG INTERSEÇÃO =====")
    print("Itens únicos preditos:", pred_df['Item'].nunique())
    print("Itens únicos no teste:", test_df['Item'].nunique())

    intersection = set(pred_df['Item']) & set(test_df['Item'])
    print("Interseção de itens:", len(intersection))
    print("Itens em comum:", intersection)

    print("\n===== DEBUG USUÁRIOS =====")
    print(test_df.groupby("User")["Item"].count())

def evaluate_model(
    pred_df,
    rec_df,
    test_df,
    label="MODEL",
    k=8,
    threshold=0.0,
    debug=False
):

    print(f"\n==============================")
    print(f"===== {label} =====")
    print(f"==============================")

    if debug:
        debug_predictions(pred_df, test_df)

    evaluate_error_metrics(pred_df, label)
    evaluate_ranking_metrics(rec_df, test_df, k, threshold, label)


def predicao(test_matrix, user_item_matrix, collab_df, user_similarity_df, item_similarity_df, method, _df, test_df):
  # 1. Garantir interseção estrutural
      print("TEST MATRIX SHAPE BEFORE:", test_matrix.shape)
      print("TRAIN MATRIX SHAPE:", user_item_matrix.shape)

      common_users = set(collab_df["User"]).intersection(test_df["User"])
      common_items = set(collab_df["Item"]).intersection(test_df["Item"])

      print("\nANTES DO FILTRO:")
      print("Test size:", len(test_df))
      print("Common users:", len(common_users))
      print("Common items:", len(common_items))

      test_df = test_df[
          test_df["User"].isin(common_users) &
          test_df["Item"].isin(common_items)
      ]

      print("\nDEPOIS DO FILTRO:")
      print("Test size:", len(test_df))

      if len(test_df) == 0:
          print("\n TEST DF VIROU VAZIO → pulando métricas")
          return


      print("TEST MATRIX SHAPE AFTER:", test_matrix.shape)


      all_items = user_item_matrix.columns

      test_df,

      rec_user_df = generate_recommendations(
          test_df,
          all_items,
          user_item_matrix,
          user_similarity_df,
          predict_cf
      )

      rec_item_df = generate_recommendations(
          test_df,
          all_items,
          user_item_matrix,
          item_similarity_df,
          predict_cf
      )

      predictions = []
      predictions_item = []

      for _, row in test_df.iterrows():

          user = row["User"]
          item = row["Item"]
          true_rating = row["Rating"]

          # ---------- USER-BASED ----------
          pred_user = predict_cf(
              user,
              item,
              user_item_matrix,
              user_similarity_df
          )

          # ---------- ITEM-BASED ----------
          pred_item = predict_cf(
              user,
              item,
              user_item_matrix,
              item_similarity_df
          )

          predictions.append({
              "User": user,
              "Item": item,
              "Rating": true_rating,
              "Prediction": pred_user
          })

          predictions_item.append({
              "User": user,
              "Item": item,
              "Rating": true_rating,
              "Prediction": pred_item
          })


      print("\n===== ITEM-BASED =====")

      pred_df = pd.DataFrame(predictions)
      pred_item_df = pd.DataFrame(predictions_item)

      evaluate_model(
          pred_df,
          rec_user_df,
          test_df,
          label="USER-BASED",
          k=8,
          threshold=0.0,
          debug=True
      )

      evaluate_model(
          pred_item_df,
          rec_item_df,
          test_df,
          label="ITEM-BASED",
          k=8,
          threshold=0.0,
          debug=False
      )


def collaborative_filtering(_df, method, teste, recomenda, df_original):
      print("-------------------- VISUALIZAÇÃO DOS DADOS ----------------------------")
      print(_df.shape)
      print(_df.head())
      print("-"*80)

      print("\nFrequência no dungeon_frame final:")
      print(_df['PreFormAnswers'].value_counts())

      # Número de usuários únicos pelo PlayerId
      num_unicos = _df['PlayerId'].nunique()
      print(f"\nNúmero de usuários únicos: {num_unicos}")

      print("-------------------- NORMALIZAÇÃO DOS DADOS ----------------------------")
      collab_df = build_collab_df(_df, method)
      test_df = build_collab_df(teste, method)

      print("\nMatriz para filtragem 10 :")
      print(test_df.head())

      print("\nMatriz para filtragem 1 :")
      print(collab_df.head())

      print("-------------------- CÁLCULOS ----------------------------")

      # Create user-item matrix
      user_item_matrix = collab_df.pivot_table(index='User', columns='Item', values='Rating', fill_value=0)
      test_matrix = test_df.pivot_table(index='User', columns='Item', values='Rating', fill_value=0)
      print(test_matrix.head())

      n_users = user_item_matrix.shape[0]
      n_items = user_item_matrix.shape[1]

      total_cells = n_users * n_items
      non_zero = (user_item_matrix > 0).sum().sum()

      sparsity = 1 - (non_zero / total_cells)

      print("\n===== SPARSITY =====")
      print("Users:", n_users)
      print("Items:", n_items)
      print("Total cells:", total_cells)
      print("Non-zero interactions:", non_zero)
      print("Sparsity:", sparsity)

      print("-------------------- VERIFICAÇÃO DOS DADOS ----------------------------")
      analyze_user_item_matrix(user_item_matrix)

      print("-------------------- SIMILIARIDADE ----------------------------")
      item_similarity_df, user_similarity_df = similiarity_pearson(user_item_matrix)
      item_similarity_df, user_similarity_df = similiarity_cosine(user_item_matrix)

      if(recomenda):
        recomendando(method, user_item_matrix, user_similarity_df, item_similarity_df, _df,df_original)


      print("-------------------- PREDIÇÃO ----------------------------")
      predicao(test_matrix, user_item_matrix, collab_df, user_similarity_df, item_similarity_df, method, _df, test_df)




def run(_df, method, recomenda):
    # Organizando o dataset
    _df['PreFormAnswers'] = _df['PreFormAnswers'].apply(lambda x: 'A=' + str(x[0]) + ", C=" + str(x[1]) + ", I=" + str(x[2]) + ", M=" + str(x[3]))
    _df = split_preform_answers(_df)
    n = _df.shape[0]
    print("_df shape:", _df.shape)

    # ANTES DO SPLIT (ESSENCIAL)
    counts = _df.groupby("PlayerId").size()
    valid_users = counts[counts >= 2].index
    _df = _df[_df["PlayerId"].isin(valid_users)]
    df_original = _df.copy()  # guarda o original antes da normalização

    # SPLIT
    train_df, test_df = leave_one_out_split(_df)

    # Normalizar
    numeric_cols = train_df.select_dtypes(include=["float64", "int64"]).columns
    cols_to_exclude = ["PlayerID", "PreFormAnswers", "LevelName"]
    numeric_cols = [c for c in numeric_cols if c not in cols_to_exclude]
    scaler = MinMaxScaler()
    train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
    test_df[numeric_cols] = scaler.transform(test_df[numeric_cols])


    # Augmentation
    _dfnoise = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n)
    _dfscale = augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)
    _dfnoise2 = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], 2*n)
    _dfscale2 = augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)
    _dfscale2 = augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.95)
    _dfmixed = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n)
    _dfmixed = pd.concat([_dfmixed, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)], ignore_index=True)
    _dfmixed10 = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], 6*n)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.95)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.10)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.9)], ignore_index=True)

    dataframes = [#_df, _dfnoise, _dfscale, _dfnoise2, _dfscale2, _dfmixed,
                  _dfmixed10]
    experiment_names = [#'Baseline', '1xN', '1xS-1_05', '2xN', '2xS-1_05+0_95', '1xS-1_05+1xN',
                        '6xN+4xS-1_05+0_95+1_1+0_9']

    strategies = [#'Original', '1xNoise', '1xScale', '2xN', '2xScale', '1xN+1xS',
                  '6xN+4xS']

    # Quantos níveis diferentes cada usuário jogou
    levels_per_user = _df.groupby("PlayerId")["LevelName"].nunique()

    # Filtrar apenas quem jogou mais de 1 nível
    multi_level_users = levels_per_user[levels_per_user > 1]

    print("Usuários que jogaram mais de um nível:")
    print(multi_level_users)

    # Filtragem colaborativa
    collaborative_filtering(_dfmixed10, method, test_df, recomenda,df_original)


def main(method, recomenda):

    initialize_data_frame()
    dungeon_frame = get_dungeon_data_frame()
    user_frame = get_user_data_frame()

    if(method == 0):
      inspect_dataset(user_frame, dungeon_frame)
    else:
      dungeon_frame = pd.merge(dungeon_frame, user_frame[['PlayerId', 'PreFormAnswers']], on=['PlayerId'], how='left')
      dungeon_frame = drop_invalid_rows(dungeon_frame)
      dungeon_frame = get_percentages(dungeon_frame)
      run(dungeon_frame, method, recomenda)


In [27]:
main(0, False)

Combinações únicas:
[(1, 3, 2, 4), (2, 1, 3, 4), (2, 1, 4, 3), (3, 2, 1, 4), (3, 2, 4, 1), (4, 1, 3, 2), (4, 2, 1, 3), (4, 3, 2, 1)]

Quantidade de combinações únicas:
8

Frequência de cada combinação:
PreFormAnswers
(4, 2, 1, 3)    3
(3, 2, 4, 1)    3
(1, 3, 2, 4)    3
(3, 2, 1, 4)    2
(4, 3, 2, 1)    1
(4, 1, 3, 2)    1
(2, 1, 3, 4)    1
(2, 1, 4, 3)    1
Name: count, dtype: int64

Frequência no dungeon_frame final:
PreFormAnswers
[1, 3, 2, 4]    26
[4, 1, 3, 2]    25
[2, 1, 3, 4]    24
[4, 2, 1, 3]    18
[3, 2, 1, 4]    11
[3, 2, 4, 1]     9
[2, 1, 4, 3]     5
[4, 3, 2, 1]     1
Name: count, dtype: int64

Total de colunas: 57

  1. CompletedAchievementQuests
  2. CompletedCreativityQuests
  3. CompletedDamageQuests
  4. CompletedExchangeQuests
  5. CompletedExploreQuests
  6. CompletedGatherQuests
  7. CompletedGiveQuests
  8. CompletedGoToQuests
  9. CompletedImmersionQuests
 10. CompletedKillQuests
 11. CompletedListenQuests
 12. CompletedMasteryQuests
 13. CompletedQuests
 14. C

In [28]:
seed = 42
random.seed(seed)
np.random.seed(seed)
main(20, True)

_df shape: (119, 67)


/tmp/ipykernel_5622/3852504427.py:79: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _data_frame.replace([np.inf, -np.inf], 0, inplace=True)


Usuários que jogaram mais de um nível:
PlayerId
53627948       4
78194049       3
81972292      11
121412094     12
282268519      4
1093542367    12
1153731241     4
1776427228     2
2079201442     4
2079942857    12
2096844718     3
Name: LevelName, dtype: int64
-------------------- VISUALIZAÇÃO DOS DADOS ----------------------------
(1710, 67)
                      PostFormAnswers  TotalEnemies  TotalMasteryQuests  MaxCombo  TotalExploreQuests  TotalNpcs  CompletedReportQuests  CompletedListenQuests  TotalDeaths  CompletedAchievementQuests  TotalGiveQuests  TotalAchievementQuests  TotalKeys  PlayerId  TotalLocks  KeysCollected  TotalExchangeQuests  TotalLostHealth  CompletedExploreQuests  CompletedExchangeQuests  TotalGoToQuests  RoomsEntered  CompletedDamageQuests  TotalAttempts  TotalCreativityQuests  TotalGatherQuests  NpcsInteracted  CompletedMasteryQuests                                       InputProfile  CompletedGiveQuests WeaponName  TotalImmersionQuests                    



## Atributos Estruturais
1. **TotalRooms** - número de salas
2. **TotalKeys** - número de chaves
3. **TotalLocks** - número de fechaduras
4. **UniqueRoomsEntered** - salas únicas visitadas
5. **RoomsEntered** - total de salas entradas (com revisitas)

## Atributos de Combate
6. **TotalEnemies** - número total de inimigos
7. **EnemiesKilled** - inimigos mortos
8. **MaxCombo** - maior combo
9. **TotalLostHealth** - vida perdida
10. **TotalDeaths** - número de mortes

## Atributos de Interação
11. **TotalNpcs** - número de NPCs
12. **NpcsInteracted** - NPCs com quem interagiu

## Atributos de Progresso
13. **TotalTreasure** - tesouros disponíveis
14. **TreasuresCollected** - tesouros coletados
15. **TimeToFinish** - tempo para finalizar

## Atributos de Missões (Quests)
16. **TotalAchievementQuests** - total de quests de conquista
17. **TotalCreativityQuests** - total de quests de criatividade
18. **TotalDamageQuests** - total de quests de dano
19. **TotalExploreQuests** - total de quests de exploração
20. **TotalGatherQuests** - total de quests de coleta
21. **TotalGiveQuests** - total de quests de entrega
22. **TotalGoToQuests** - total de quests de "ir até"
23. **TotalImmersionQuests** - total de quests de imersão
24. **TotalKillQuests** - total de quests de matar
25. **TotalListenQuests** - total de quests de ouvir
26. **TotalMasteryQuests** - total de quests de mestria
27. **TotalReadQuests** - total de quests de ler
28. **TotalReportQuests** - total de quests de relatório

## Métricas Calculadas
29. **LockUsage** - uso de fechaduras (LocksOpened/TotalLocks)
30. **KeyCollection** - coleta de chaves (KeysCollected/TotalKeys)
31. **MapCompletion** - exploração (UniqueRoomsEntered/TotalRooms)
32. **RoomRevisitRate** - taxa de revisita
33. **EnemyKillRate** - taxa de abate (EnemiesKilled/TotalEnemies)
34. **Success Rate** - taxa de sucesso (Wins/Attempts)

